## Imports

In [1]:
%load_ext autoreload
%autoreload 2

import urllib.request
api_url = 'https://raw.githubusercontent.com/tanmayyb/ele70_bv03/refs/heads/main/api/datasets.py'
exec(urllib.request.urlopen(api_url).read())

In [2]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error

## Trainer Code

In [5]:
dataset_types = ['zonal', 'fsa']
datetimes = {'zonal': ('2018', '2020'),
            'fsa': ('201801', '202001')}

def mean_absolute_percentage_error(y_true, y_pred):
    """Calculates MAPE given y_true and y_pred"""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

def train_and_evaluate():
  global dataset_types, datetimes
  results = {}
  for dataset_type in dataset_types:
    results[dataset_type] = {}
    start_date, end_date = datetimes[dataset_type]
    for target_val in range(5):
      ieso = IESODataset(dataset_type)
      ieso.set_target(target_val)
      ieso.load_dataset(start_date=start_date, end_date=end_date, download=True)

      climate = ClimateDataset(ieso)
      climate.load_dataset(sample_num=5, download=True)

      preprocessor = DatasetPreprocessor(ieso, climate)
      target_name, dataset, dt = preprocessor.preprocess()
      (X_train, X_test, y_train, y_test), (train_idx, test_idx) = create_train_test_split(dataset, target=target_name, dt=dt)

      reg = xgb.XGBRegressor(n_estimators=1000)
      reg.fit(X_train, y_train,
              eval_set=[(X_train, y_train), (X_test, y_test)],
              # early_stopping_rounds=50,
            verbose=False)
      
      pred = reg.predict(X_test)

      results[dataset_type][target_val] = mean_absolute_percentage_error(y_test, pred)

  return results

results = train_and_evaluate()

Available years: 2003 to 2024
Selected Time Range for IESO zonal dataset: 2018 to 2020


Processing chunks: 100%|██████████| 1/1 [00:00<00:00,  8.81it/s]


Available years: 2003 to 2024
Selected Time Range for IESO zonal dataset: 2018 to 2020


Processing chunks: 100%|██████████| 1/1 [00:00<00:00,  8.91it/s]


Available years: 2003 to 2024
Selected Time Range for IESO zonal dataset: 2018 to 2020


...

Processing chunks: 100%|██████████| 7/7 [00:24<00:00,  3.52s/it]


Available Time range for IESO fsa dataset: 201801 to 202411
Selected Time Range for IESO fsa dataset: 201801 to 202001


Processing chunks: 100%|██████████| 7/7 [00:22<00:00,  3.15s/it]


Available Time range for IESO fsa dataset: 201801 to 202411
Selected Time Range for IESO fsa dataset: 201801 to 202001


Processing chunks: 100%|██████████| 7/7 [00:22<00:00,  3.26s/it]


In [13]:
pd.DataFrame(results).mean(axis=0)

zonal    8.825103
fsa      7.626740
dtype: float64

In [14]:
100.0 - pd.DataFrame(results).mean(axis=0)

zonal    91.174897
fsa      92.373260
dtype: float64

In [ ]:
IESO_DATASET.get_dataset_info()

In [10]:
ieso_dataset = IESODataset('fsa')
ieso_dataset.get_target_options()


Available Time range for IESO fsa dataset: 201801 to 202411


['Toronto',
 'Ottawa',
 'Hamilton',
 'Mississauga',
 'Brampton',
 'Kitchener',
 'London',
 'Markham',
 'Oshawa',
 'Vaughan',
 'Windsor',
 'St. Catharines',
 'Oakville',
 'Richmond Hill',
 'Burlington',
 'Sudbury',
 'Barrie',
 'Guelph',
 'Whitby',
 'Cambridge',
 'Milton',
 'Ajax',
 'Waterloo',
 'Thunder Bay',
 'Brantford',
 'Chatham',
 'Clarington',
 'Pickering',
 'Niagara Falls',
 'Newmarket',
 'Peterborough',
 'Kawartha Lakes',
 'Caledon',
 'Belleville',
 'Sarnia',
 'Sault Ste. Marie',
 'Welland',
 'Halton Hills',
 'Aurora',
 'North Bay',
 'Stouffville',
 'Cornwall',
 'Georgina',
 'Woodstock',
 'Quinte West',
 'St. Thomas',
 'New Tecumseth',
 'Innisfil',
 'Bradford West Gwillimbury',
 'Timmins',
 'Lakeshore',
 'Brant',
 'Leamington',
 'East Gwillimbury',
 'Orangeville',
 'Orillia',
 'Stratford',
 'Fort Erie',
 'LaSalle',
 'Centre Wellington',
 'Grimsby',
 'King',
 'Woolwich',
 'Clarence-Rockland',
 'Midland',
 'Lincoln',
 'Wasaga Beach',
 'Collingwood',
 'Strathroy-Caradoc',
 'Thorold

## Results

#### XGB

In [21]:
pd.DataFrame(results)

,zonal,fsa
0,11.088679,7.700692
1,6.329063,6.082491
2,9.165442,5.980998
3,8.832689,7.335450
4,8.709643,11.034067


#### LSTM

In [15]:
lstm_zonal_index = ['Northwest', 'Northeast', 'Ottawa', 'East', 'Toronto']
lstm_zonal_mape = [2.17, 2.31, 2.34, 3.87, 1.54]

lstm_fsa_index = ['Toronto', 'Ottawa', 'Hamilton', 'Mississauga', 'Brampton']
lstm_fsa_mape = [4.65, 6.07, 4.04, 4.86, 4.40]


In [24]:
pd.DataFrame({'zonal': pd.Series(lstm_zonal_mape, index=lstm_zonal_index).reset_index(drop=True),
              'fsa': pd.Series(lstm_fsa_mape, index=lstm_fsa_index).reset_index(drop=True)}).mean(axis=0)

zonal    2.446
fsa      4.804
dtype: float64